In [20]:
#Pasos para subir archivos desde Colab a Repo en GitHub

#Configuración para repo GITHUB
!git config --global use.name "Heydrian"
!git config --global user.email "adrianantunezcruz@gmail.com"

#Clonamos repo en Colab
from google.colab import userdata
USER = "Heydrian"
TOKEN = userdata.get("GITHUB_TOKEN")
REPO = "Public-Sources-Analysis-applied-AED-Statistic-Analysis-and-Machine-Learning"

#Comando específico para clonar
!git clone http://{TOKEN}@github.com/{USER}/{REPO}.git




fatal: destination path 'Public-Sources-Analysis-applied-AED-Statistic-Analysis-and-Machine-Learning' already exists and is not an empty directory.


In [ ]:
import json
from google.colab import _message

#nos ubicamos en la carpeta del repo
%cd "/content/Public-Sources-Analysis-applied-AED-Statistic-Analysis-and-Machine-Learning"

# Guarda el notebook actual directamente dentro de la carpeta del repositorio
with open("ML Electric Car Market.ipynb", "w", encoding="utf-8") as f:
    # Obtenemos el JSON de tu notebook actual
    notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
    json.dump(notebook_json['ipynb'], f, indent=2)

print("Archivo guardado físicamente en la carpeta del repositorio.")

In [ ]:
#Subimos archivo a GitHub
!git add "ML Electric Car Market.ipynb"

!git commit -m "Colab Notebook con análisis de mercado de carros eléctricos."

!git push origin main

# **PROYECTO: ANÁLISIS DE MERCADO DE VEHÍCULOS ELÉCTRICOS**

# **Sobre este proyecto:**



El presente documento representa la integración de conocimientos adquiridos durante el curso "Master en Data Science con IA".

El proyecto consta de los siguientes puntos principales:

* Data Ingestion: identificación y extracción de fuentes de datos.
* Data wrangling: limpieza y procesamiento de datos.
* EDA: Análisis exploratorio para identificar patrones y tendencias mediante estadística descriptiva, correlaciones y detección de valores atípicos.
* Análisis profundo: realización de pruebas y modelado de algoritmos para:
  * Definir alcance del análisis.
  * Identificar tendencias en ventas, producción y del mercado de vehículos eléctricos.
  * Análisis del tamaño del mercado y tasas de crecimiento para diferentes sectores de EV.
  * Recomendaciones estratégicas para empresas que busquen ingresar o expandirse en este mercado.
* Interpretación de datos: creación de una narrativa visual para exponer los insights de la investigación

#Data Ingestion (Identificación y extracción de información)

Para este proyecto su usará la base de datos que se puede descargar [aquí](https://docs.google.com/spreadsheets/d/1EvSrAe5JUoDUjU4x8Mm43q41Zecnov0wL4h8BbOT18M/edit?usp=drive_link).

Esta base representa información de vehículos eléctricos registradosa por el Washington State Department of Licensing, DOL, parte de la base pública del gorbierno de USA.

##Configuración de entorno
Importamos las librerías necesarias.

In [ ]:
import re
import unicodedata

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.cluster import MiniBatchKMeans
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import silhouette_score

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

Descargamos y guardamos el dataset.

In [ ]:
# URL para la exportación a CSV
sheet_id = '1EvSrAe5JUoDUjU4x8Mm43q41Zecnov0wL4h8BbOT18M' #identificador general
gid = '86233506' #Identificador de la hoja

# URL completa para descargar el CSV desde Google Sheets
sheet_url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'

# Lectura directa en la memoria del entorno
df_electric_cars = pd.read_csv(sheet_url)

# Verificación de la estructura del dataset
print(df_electric_cars.info())

# Data Wrangling: Manipulación y saneamiento de los datos
* Correción y/o imputación de valores nulos.
* Normalización de formatos de texto.
* Transformación de tipos de datos.

##Inspección de valores nulos

In [ ]:
# Definimos función para calcular los na
def obtener_nulos(df):
    """
    Cuenta los valores nulos de un DataFrame, calcula su porcentaje
    y devuelve un DataFrame con las columnas que tienen nulos.
    """
    # Contar los nulos por columna y convertir el resultado en un DataFrame
    nulos_df = df.isna().sum().to_frame("Valores nulos")

    # Calcular el porcentaje de nulos basado en el total de filas (shape[0])
    nulos_df["Porcentaje nulos"] = nulos_df["Valores nulos"] / df.shape[0]

    # Formatear el porcentaje a string con 3 decimales y el símbolo "%"
    nulos_df["Porcentaje nulos"] = nulos_df["Porcentaje nulos"].apply(lambda x: f"{x * 100:.03f}%")

    # Filtrar para mantener solo las columnas que tienen al menos 1 valor nulo
    nulos_df = nulos_df[nulos_df["Valores nulos"] > 0]

    # 5. Retornar el DataFrame resultante
    return nulos_df

# Calculamos los valores nulos para nuestro dataframe
obtener_nulos(df_electric_cars)

In [ ]:
#Observamos las filas con nulos
df_electric_cars[df_electric_cars["City"].isna()]

In [ ]:
#Eliminamos los valores na de la columna "City"
df_electric_cars_limpio = df_electric_cars.dropna(subset=["City"]).copy()

#Obtenemos la cantidad de nulos
obtener_nulos(df_electric_cars_limpio)

**Explicación:**
Se eliminaron las filas con valores nulos de la variable "City" por las siguientes razones:
* Existía un patrón en los valores nulos: correspondían a países externos a USA. Claves: AE = Armed Forces Europe, AP = Armed Forces Pacific, BC = British Columbia, Canadá.
* Las filas compartían nulos en otras columnas como County, Post Code, Legislative District, Vehicle Location, Electric Utility, 2020 Census Tract. Datos geográficos que, al no estar presentes, no nos permiten reconstruir la ubicación de la ciudad.
* El porcentaje de los nulos era una cifra despreciable (0.003%).

**Resultado:** obtenemos una base de datos filtrado sólo regionalmente, acotada a los Estados Unidos de América.

Para el caso de los na en **"Vehicle Location"** (0.002%), se imputará por medio de la reconstrucción de la localización a partir de centroides espaciales calculados con el Census Track y el Vehicule Location.
Esto se logra comparando Tracks idénticos y calculando distancias en las coordenadas del Vehicle Location compartidas por Tracks iguales.

In [ ]:
# ————— PASO 1: EXTRACCIÓN DE COORDENADAS —————

# Extraer lat/lon de POINT a columnas separadas
def extraer_coordenadas(point_str):
    # Si el valor es nulo, detenemos la función aquí y devolvemos NaN
    if pd.isna(point_str):
        return np.nan, np.nan
    # Reemplazamos valores POINT, paréntesis y separamos por espacios
    coords = point_str.replace("POINT (", "").replace(")", "").split()
    latitude = float(coords[1]) #Tomamos la latitud de la posición [1]
    longitude = float(coords[0])  #Tomamos la latitud de la posición [0]
    return latitude, longitude

# Aplicamos la función para crear las columnas numéricas de longitud y latitud
df_electric_cars_limpio[["Latitude", "Longitude"]] = df_electric_cars_limpio["Vehicle Location"].apply(
    lambda x: pd.Series(extraer_coordenadas(x)) # Devuleve una serie pandas con valores de las coordenadas
)

# ————— PASO 2: CÁLCULO DE CENTROIDES —————

# Calculamos los centroides (promedio de coordenadas por cada sector censal)
centroides = df_electric_cars_limpio.groupby("2020 Census Tract")[["Longitude", "Latitude"]].mean()

# Creamos una máscara booleana con las filas donde "Vehicle Location" es nulo
mask = df_electric_cars_limpio["Vehicle Location"].isna()

# ————— PASO 3: IMPUTACIÓN DE DATOS —————
# Rellenamos los vacíos usando el mapa de centroides
df_electric_cars_limpio.loc[mask, "Vehicle Location"] = df_electric_cars_limpio.loc[mask, "2020 Census Tract"].map(
    lambda x: f"POINT ({round(centroides.loc[x, "Longitude"], 5)} {round(centroides.loc[x, "Latitude"], 5)})" #Aplicación de mapeo de valores obtenidos por centroides
    if pd.notna(x) and x in centroides.index #Control para valores nana
    else np.nan
)

#Eliminamos las columnas dummy utlizadas para calcular nuestros valores imputados
df_electric_cars_limpio = df_electric_cars_limpio.drop(columns=["Latitude", "Longitude"], axis=1)

In [ ]:
# Validación de Formato: Ver un ejemplo de los datos ya rellenos
print("————— EJEMPLO DE FORMATO ACTUALIZADO —————")
print(df_electric_cars_limpio.loc[mask, ["2020 Census Tract", "Vehicle Location"]].head())

# Validación de Precisión: Comparar una fila rellena contra su centroide real
print("\n————— COMPROBACIÓN DE PRECISIÓN —————")
ejemplo_tract = df_electric_cars_limpio.loc[mask, "2020 Census Tract"].iloc[0]
print(f"Para el Tract '{ejemplo_tract}':")
print("Centroide calculado originalmente (Lon, Lat):", centroides.loc[ejemplo_tract].values)
print("Texto generado en el DataFrame:            ", df_electric_cars_limpio.loc[mask, "Vehicle Location"].iloc[0])

In [ ]:
#Realizamos nuevamente el cálculo de nulos
obtener_nulos(df_electric_cars_limpio)

Imputamos los datos de "Legislative District".

Realizaremos imputaciones geográficas para determinar el distrio legislativo al que pertenecen los registros mediante la comparación de los Census Tracts.

In [ ]:
# Creamos un diccionario que asocie cada "2020 Census Tract" con su distrito más común (moda)
tract_a_distrito = df_electric_cars_limpio.groupby("2020 Census Tract")["Legislative District"].apply(
    lambda x: x.mode().iloc[0] #Calcula la moda: número de distrito legislativo que más se repite
    if not x.mode().empty #validación de seguridad si el cálculo de la moda arrojó algún resultado
    else np.nan
).to_dict()

# Identificamos las filas donde el distrito es nulo
mask_distrito = df_electric_cars_limpio["Legislative District"].isna()

# Imputamos el distrito buscando el "2020 Census Tract" en nuestro diccionario
df_electric_cars_limpio.loc[mask_distrito, "Legislative District"] = df_electric_cars_limpio.loc[mask_distrito, "2020 Census Tract"].map(tract_a_distrito)

# Los registros que queden (si queda alguno) se pueden rellenar con la moda global y pasamos a entero
global_mode = df_electric_cars_limpio["Legislative District"].mode().iloc[0]
df_electric_cars_limpio["Legislative District"] = df_electric_cars_limpio["Legislative District"].fillna(global_mode).astype(int)

In [ ]:
#Isnpección de valores nulos
obtener_nulos(df_electric_cars_limpio)

## Limpieza de variables de texto
Regularización de valores:
* Aplicamos regex para estandarizar a un texto en mayúsculos sin símbolos especiales.
* Cambiamos las claves de los estados para mejorar el entendimiento de los valores de la variable State


In [ ]:
#Definimos función para normalizar
def normalizar_texto(texto):
    """
    Convierte a mayúsculas, elimina acentos, remueve caracteres especiales,
    pero conserva espacios, guiones normales y paréntesis.
    Lo que se conserva es parte del formato original de los datos y lo que se elimina son posibles typos y errores.
    """
    if pd.isna(texto):
        return np.nan

    # Convierte a string, mayúsculas y quita espacios al inicio y final
    texto = str(texto).upper().strip()

    # Elimina acentos o diéresis (ej: Á -> A)
    texto = "".join(
        #Toma cada caracter acentuado y los separa
        c for c in unicodedata.normalize("NFD", texto) #NFC acento y letra como dos componentes separados
        if unicodedata.category(c) != "Mn"
    )

    # Remueve caracteres especiales
    # Conserva: Letras, Números, Espacios (\s), Paréntesis (\(\)) y el guion al final (-)
    texto = re.sub(r"[^A-Z0-9\s\(\)-]", "", texto)

    return texto

In [ ]:
#Normalizamos nuestras variables de texto
col_texto = df_electric_cars_limpio.select_dtypes(include = ["object", "category"]).columns
df_electric_cars_limpio[col_texto] = df_electric_cars_limpio[col_texto].apply(lambda x: x.map(normalizar_texto))

In [ ]:
#Reemplazamos las claves de estados por sus nombres
mapa_estados = {
    'AK': 'ALASKA', 'AL': 'ALABAMA', 'AR': 'ARKANSAS', 'AZ': 'ARIZONA',
    'CA': 'CALIFORNIA', 'CO': 'COLORADO', 'CT': 'CONNECTICUT', 'DC': 'DISTRICT OF COLUMBIA',
    'DE': 'DELAWARE', 'FL': 'FLORIDA', 'GA': 'GEORGIA', 'HI': 'HAWAII',
    'IA': 'IOWA', 'ID': 'IDAHO', 'IL': 'ILLINOIS', 'IN': 'INDIANA',
    'KS': 'KANSAS', 'KY': 'KENTUCKY', 'LA': 'LOUISIANA', 'MA': 'MASSACHUSETTS',
    'MD': 'MARYLAND', 'ME': 'MAINE', 'MI': 'MICHIGAN', 'MN': 'MINNESOTA',
    'MO': 'MISSOURI', 'MS': 'MISSISSIPPI', 'MT': 'MONTANA', 'NC': 'NORTH CAROLINA',
    'ND': 'NORTH DAKOTA', 'NE': 'NEBRASKA', 'NH': 'NEW HAMPSHIRE', 'NJ': 'NEW JERSEY',
    'NM': 'NEW MEXICO', 'NV': 'NEVADA', 'NY': 'NEW YORK', 'OH': 'OHIO',
    'OK': 'OKLAHOMA', 'OR': 'OREGON', 'PA': 'PENNSYLVANIA', 'RI': 'RHODE ISLAND',
    'SC': 'SOUTH CAROLINA', 'SD': 'SOUTH DAKOTA', 'TN': 'TENNESSEE', 'TX': 'TEXAS',
    'UT': 'UTAH', 'VA': 'VIRGINIA', 'VT': 'VERMONT', 'WA': 'WASHINGTON',
    'WI': 'WISCONSIN', 'WV': 'WEST VIRGINIA', 'WY': 'WYOMING'
}

#Mapeamos valores
df_electric_cars_limpio["State"] = df_electric_cars_limpio["State"].map(mapa_estados)

#Nos aseguramos que el formato final del texto sea normalizado
df_electric_cars_limpio["State"] = df_electric_cars_limpio["State"].map(normalizar_texto)

In [ ]:
#Inspeccionamos nuestro dataset para corroborar los cambios realizados
df_electric_cars_limpio.head(3)

Transformamos nuestras variables categóricas:
* category para categorías y/o clases grupales.
* int para claves, identificadores, códigos y datos discretos.
* object para descricpiones, claves e identificadores alfanuméricas.
* datetime: para este caso, se opta por manejar la variable Model Year como varaible int. Esto debido a que no existen datos día-mes para reconstruirla adecuadamente. Además, el uso como int nos permite hacer análisis con cálculos matemáticos.

**Nota:** La variable VIN (1-10) es el número de identificación vehicular (Vehicle Identification Number), el cual es un código de 17 caracteres alfanuméricos. Se opta por tomarlo como variable objeto.

In [ ]:
#Creamos una lista con los nombres de las variables separadas por tipos
var_categ = ["County","City", "State","Make", "Model", "Electric Vehicle Type", "Clean Alternative Fuel Vehicle (CAFV) Eligibility","Electric Utility"]

#Por manejor de claves geográficas, Postal Code y 2020 Census Track permanecen como float para no afectar su formato con ceros.
var_int = ["Model Year", "Electric Range","Base MSRP","Legislative District"]

#Convertimos al tipo adecuado
df_electric_cars_limpio[var_int] = df_electric_cars_limpio[var_int].astype("int")
df_electric_cars_limpio[var_categ] = df_electric_cars_limpio[var_categ].astype("category")


df_electric_cars_limpio.info()

## Limpieza de variables numéricas
Inspección de valores:
* Valores atípicos y valores extremos.

Se explican las siguientes variables numéricas:
* Base MSRP: Manufacturer's Suggested Retail Price (Precio de Venta Sugerido por el Fabricante)o el Precio de Lista o Precio Base de Venta al Público.
  * Contiene 0 que indican fluctuaciones en precio, versiones VIN sin coincidencia en las bases de precios o desactualizaciones en las bases del MSRP, lo que provoca que la variable se registre como 0. No se recomienda imputar ni eliminar para no perder información ni alterar cálculo. Se opta por un manejo filtrado de información.
* Electric Range (Autonomía Eléctrica): distancia máxima estimada (en millas) que el vehículo puede recorrer utilizando exclusivamente la energía almacenada en su batería de tracción antes de requerir una recarga conectada a la red o de que se encienda el motor de combustión interna.

* DOL Vehicle ID (Identificador de Vehículo del DOL): clave numérica interna asignada de forma secuencial y única por el Department of Licensing (DOL) del Estado de Washington.

* Model Year: año de prouducción del modelo del vehículo.

Para esta sección se prescindirán de las variables Postal Code, 2020 Census Tract, Legislative District y DOL Vehicle ID, las cuales, aunque son variables numéricas, representan códigos geográficos, administrativo-políticos o de control interno de expedientes.
Por ello, sólo se eligen variables numéricas suceptibles de ser revisadas bajo una exploración estadística general para detección de outliers.

In [ ]:
# Lista de variables cuantitativas seleccionadas
var_analisis_num = ["Model Year", "Electric Range", "Base MSRP"]

# Configuración estética global
plt.style.use('dark_background')
sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "#121212",
    "figure.facecolor": "#121212",
    "grid.color": "#292929",       # Mantenemos la cuadrícula sutil para no saturar
    "text.color": "#FFFFFF",       # Blanco puro para textos generales
    "axes.labelcolor": "#FFFFFF"   # Blanco puro para los títulos de los ejes
})

palette = sns.color_palette("viridis", as_cmap=False)
color_principal = palette[3]

# Iteración y graficación
for var in var_analisis_num:

    # Calcular métricas estadísticas para la variable actual
    media = df_electric_cars_limpio[var].mean()
    mediana = df_electric_cars_limpio[var].median()

    fig, (ax_hist, ax_box) = plt.subplots(
        2, 1,
        figsize=(10, 6),
        sharex=True,
        gridspec_kw={"height_ratios": (0.7, 0.3)}
    )

    # ————— GRÁFICO 1: HISTOGRAMA —————
    sns.histplot(
        data=df_electric_cars_limpio,
        x=var,
        ax=ax_hist,
        kde=True,
        color=color_principal,
        alpha=0.6,
        line_kws={"linewidth": 2}
    )
    ax_hist.set_title(f'Distribución de : {var}', fontsize=14, pad=15, fontweight='bold', color='#FFFFFF')
    ax_hist.set_ylabel('Cantidad de observaciones', fontsize=11, color='#FFFFFF')

    # ————— GRÁFICO 2: BOXPLOT —————
    sns.boxplot(
        data=df_electric_cars_limpio,
        x=var,
        ax=ax_box,
        color=color_principal,
        #Personalización de outliers
        flierprops={
            "marker": "o",
            "markerfacecolor": "#FF5555",
            "markeredgecolor": "none",
            "markersize": 5
        },
        #Alcance de los bigotes
        whis=1.5
    )
    ax_box.set_xlabel(var, fontsize=11, fontweight='bold', color='#FFFFFF')

    # ————— AÑADIR LÍNEAS DE MEDIA Y MEDIANA —————
    # Se añaden a ambos ejes (ax_hist y ax_box) para mantener la simetría visual
    for ax in [ax_hist, ax_box]:
        # Línea de la Media (Línea discontinua amarilla)
        ax.axvline(x=media, color='#FFCC00', linestyle='--', linewidth=2, label=f'Media: {media:.2f}')
        # Línea de la Mediana (Línea continua cian)
        ax.axvline(x=mediana, color='#00E5FF', linestyle='-', linewidth=2, label=f'Mediana: {mediana:.2f}')

    # Añadir leyenda solo al histograma para evitar duplicidad y mantener limpio el diseño
    ax_hist.legend(facecolor='#121212', edgecolor='#FFFFFF', loc='upper right')

    # ————— AJUSTE EXPLÍCITO DE CONTRASTE PARA LOS EJES —————
    # Forzamos que los números de las escalas (ticks) de ambos gráficos sean blancos
    ax_hist.tick_params(colors='#FFFFFF', labelsize=10)
    ax_box.tick_params(colors='#FFFFFF', labelsize=10)

    # Forzamos que las líneas del marco del gráfico también sean blancas para dar estructura
    for ax in [ax_hist, ax_box]:
        for spine in ax.spines.values():
            spine.set_color('#FFFFFF')
            spine.set_linewidth(0.5)

    plt.tight_layout()
    plt.show()



In [ ]:
#Obtenemos los datos exactos de:
  #Cantidad y proporción de números 0 en cada varaible numérica
  #Cantidad y proporción de valores atípcios
df_electric_cars_limpio['Range_Is_Zero'] = df_electric_cars_limpio['Electric Range'] == 0
df_electric_cars_limpio['MSRP_Is_Zero'] = df_electric_cars_limpio['Base MSRP'] == 0

# 2. Obtener las frecuencias absolutas (conteo de registros con cero) por año
cant_range_zero = df_electric_cars_limpio.groupby('Model Year')['Range_Is_Zero'].sum()
cant_msrp_zero = df_electric_cars_limpio.groupby('Model Year')['MSRP_Is_Zero'].sum()
total_por_año = df_electric_cars_limpio.groupby('Model Year').size()

# 3. Calcular las proporciones porcentuales respecto al total de vehículos de cada año
prop_range_zero = (cant_range_zero / total_por_año) * 100
prop_msrp_zero = (cant_msrp_zero / total_por_año) * 100

# 4. Consolidar los vectores en la Tabla de Contingencia Estructurada
tabla_contingencia = pd.DataFrame({
    'Total Registros': total_por_año,
    'Cant. Range == 0': cant_range_zero,
    '% Range == 0': prop_range_zero.round(2),
    'Cant. MSRP == 0': cant_msrp_zero,
    '% MSRP == 0': prop_msrp_zero.round(2)
})

# 5. Formatear estéticamente el índice
tabla_contingencia.index.name = 'Año del Modelo'

# 6. Desplegar la matriz de resultados ordenada cronológicamente
print("--- TABLA DE CONTINGENCIA: ANÁLISIS DE CEROS POR AÑO ---")
print(tabla_contingencia.sort_index())


#Cálculo del número y porcentaje de valores 0 en Electric Range
porcentaje_ceros_range = (df_electric_cars_limpio["Electric Range"] == 0).mean() * 100

print("\n———— INFORMACIÓN DE LA VARIABLE ELECTRIC RANGE ————")
print(f"\nPorcentaje de ceros en Electric Range: {porcentaje_ceros_range:.02f}%")
print(f"Número de ceros en Electric Range: {(df_electric_cars_limpio['Electric Range'] == 0).sum()}")

#Cálculo del número y porcentajes de atípicos por iqr
q1 = df_electric_cars_limpio['Base MSRP'].quantile(0.25)
q3 = df_electric_cars_limpio['Base MSRP'].quantile(0.75)
iqr = q3 - q1

limite_superior = q3 + (1.5 * iqr)
limite_inferior = q1 - (1.5 * iqr)

atipico = (df_electric_cars_limpio["Base MSRP"] > 0) & (
    (df_electric_cars_limpio["Base MSRP"] < limite_inferior) |
    (df_electric_cars_limpio["Base MSRP"] > limite_superior)
)

porcentaje_atipicos_msrp = atipico.mean() * 100
num_ceros_msrp = (df_electric_cars_limpio["Base MSRP"] == 0).sum()
porcentaje_cero_msrp = (num_ceros_msrp / len(df_electric_cars_limpio)) * 100
#Descripción de datos de variable Base MSRP
print("\n———— INFORMACIÓN DE LA VARIABLE BASE MSRP ————")
print(f"Número de valores atípicos en Base MSRP: {(atipico).sum()}")
print(f"Número de ceros en Base MSRP: {(df_electric_cars_limpio['Base MSRP'] == 0).sum()}")
print(f"Porcentaje de 0 en Base MSRP: {porcentaje_cero_msrp:.2f}%")


**Aplicación de criterios de saneamiento**

La variable **Model Year** no sufrirá imputaciones o eliminaciones de datos. Aunque presenta una concentración desproporcionada en años recientes, esto se debe a un flujo histórico de inserción de los vehículos eléctricos en el mercado. Para un análisis posterior se filtrará en años de estabilidad comercial para periodos más recientes.  

Para el caso de **Electric Range**, un valor de 0 no indica que el vehículo carezca de batería o que sea un auto de combustión interna tradicional. Todos los vehículos de la base de datos son, por definición de la normatividad de registro del Estado de Washington, vehículos eléctricos puros (BEV, Battery Electric Vehicle) o híbridos enchufables (PHEV, Plug-in Hybrid Electric Vehicle).
Por lo tanto, se buscará recrear el valor de la autonomía (Electric Range) mediante la utilización de extracción de valores por grupos (imputación por vecindad) que compartan características similares (Make, Model, Model Year).

Caso de la variable **MSRP**: es una situación similar que Electric Range, pero agravada, ya que la cifra de 0 registrados es casi un 98% del dataset. No se recomienda imputarla ni utilizarla para cálculo de coeficientes, tasas o modelados, ya que genera demasiado ruido y puede afectar nuestros resultados.

**NOTA:** Gracias a la tabla cruzada, previamente mostrada, entre Model Year, Electric Range y MSRP se notará un patrón contundente: la inmensa mayoría de los vehículos con rango cero corresponden a modelos de 2021 en adelante, esto debido a temas de registro burocrático, por lo que se opta por no elminar valores 0 para evitar una enorme **pérdida** de información **(≈51%)**.



Imputación de valores de Electric Range por grupos


In [ ]:
# ————— Fase 1: Declaración de Nulos y Segmentación Inicial —————
# Reemplazamos valores 0 con na para imputar nuestros datos
df_electric_cars_limpio['Electric Range'] = df_electric_cars_limpio['Electric Range'].replace(0, np.nan)

# Creamos una máscara booleana para identificar aquellos valores que no sean na
df_electric_cars_limpio['Range_Known'] = df_electric_cars_limpio['Electric Range'].notna()

# ————— Fase 2: Nivel de Imputación 1 (Marca + Modelo + Año) —————
# Obtenemos la mediana obtenidos por agrupación
rango_por_modelo = (df_electric_cars_limpio[df_electric_cars_limpio['Range_Known']].groupby(['Make', 'Model', 'Model Year'], observed=False)['Electric Range'].median())

# Creamos la columna imputada
df_electric_cars_limpio['Electric Range Imputed'] = df_electric_cars_limpio["Electric Range"]

#Creamos una mascara booleana para identificar los na
mask = df_electric_cars_limpio['Electric Range'].isna()

# Imputamos los valores na con una descomposición del multi-index para obtener la media para cada tupla Make-Model-Model Year
# map: para cada combinación coincidente del grupo, aplica las medianas calculadas
df_electric_cars_limpio.loc[mask, 'Electric Range Imputed'] = df_electric_cars_limpio.loc[mask, ['Make', 'Model', 'Model Year']].apply(tuple, axis=1).map(rango_por_modelo)

# ————— Fase 3: Nivel de Imputación 2 (Marca + Modelo) —————
#Creamos un objeto que contenga las medianas por grupos Make-Model
rango_por_modelo_sin_year = (df_electric_cars_limpio[df_electric_cars_limpio['Range_Known']]
   .groupby(['Make', 'Model'])['Electric Range'].median())


mask = df_electric_cars_limpio['Electric Range Imputed'].isna()

#Mapeo de tuplas para imputar valores na con medianas
df_electric_cars_limpio.loc[mask, 'Electric Range Imputed'] = (df_electric_cars_limpio[mask]
   .set_index(['Make', 'Model']).index.map(rango_por_modelo_sin_year))

# ————— Fase 4: Nivel de Imputación 3 — Respaldo Estadístico Global por Categoría —————
# Mediana por tipo de auto
mediana_bev = df_electric_cars_limpio[
    # Sólo coumnas que contenga "BEV" en su texto (BATTERY ELECTRIC VEHICLE)
    (df_electric_cars_limpio['Electric Vehicle Type'].str.contains('BEV')) &
    (df_electric_cars_limpio['Electric Vehicle Type'].str.contains('BEV')) &
    (df_electric_cars_limpio['Range_Known'])
]['Electric Range'].median()

df_electric_cars_limpio.loc[
    (df_electric_cars_limpio['Electric Vehicle Type'].str.contains('BEV')) &
    (df_electric_cars_limpio['Electric Range Imputed'].isna()),
    'Electric Range Imputed'
] = mediana_bev

# ————— Fase 5: Auditoría de Control de Calidad —————
# Corroboramos que los valores nulos hayan sido impuados
print(df_electric_cars_limpio['Electric Range Imputed'].isna().sum())


Graficamos para comparar las distribuciones de los valores no imputados e imputados de Electric Range.

**Nota:** la columna original no contempla el 51% de los datos que fueron convertidos de 0 a NAN.

**Alternativa metodológica:** Este proceso de imputación también se puede realizar por algoritmos ML como KNN o Decision Tree

In [ ]:
# Crear la figura y definir dimensiones
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# SUBGRÁFICO 1: Distribución Original (Con Ceros/NaN)
sns.histplot(
    data=df_electric_cars_limpio,
    x="Electric Range",
    kde=True,
    ax=axes[0],
    color="#FF5A5F",       # Tono coral para indicar datos crudos/con anomalías
    alpha=0.6,
    bins=30
)
axes[0].set_title("Distribución Original: Electric Range", fontsize=13, pad=15, fontweight='bold')
axes[0].set_xlabel("Autonomía (Millas)", fontsize=11, labelpad=10)
axes[0].set_ylabel("Frecuencia (Absoluta)", fontsize=11, labelpad=10)

# SUBGRÁFICO 2: Distribución Post-Imputación (Pipeline Cascada)
sns.histplot(
    data=df_electric_cars_limpio,
    x="Electric Range Imputed",
    kde=True,
    ax=axes[1],
    color="#00A699",       # Tono esmeralda para indicar datos limpios/procesados
    alpha=0.6,
    bins=30
)
axes[1].set_title("Distribución Final: Electric Range Imputed", fontsize=13, pad=15, fontweight='bold')
axes[1].set_xlabel("Autonomía Imputada (Millas)", fontsize=11, labelpad=10)
axes[1].set_ylabel("Frecuencia (Absoluta)", fontsize=11, labelpad=10)

# AJUSTES FINOS DE MAQUETACIÓN
# Forzar explícitamente el color blanco en las etiquetas numéricas de los ejes
for ax in axes:
    ax.tick_params(axis='both', colors='#FFFFFF', labelsize=10)

# Título principal de la composición arquitectónica de gráficos
fig.suptitle(
    "Impacto Estadístico de la imputación en la Autonomía Vehicular",
    fontsize=16,
    fontweight='bold',
    color='#FFFFFF',
    y=0.98
)

plt.tight_layout()
plt.show()

**Resumen:**
Después de un proceso de imputación estructurada por grupos y mediana, obtenemos una variable reconstruida con validez sólida para representar una gran parte del dataset.
Esto nos permite realizar mejores cálculos de relaciones entre variables.

# EDA Análisis Exploratorio  para hallar patrones y relaciones
En este bloque se revisarán los siguientes puntos:

* Utilizar datos históricos para identificar tendencias en ventas, producción y mercado de EV*.

* Analisis del tamaño del mercado y las tasas de crecimiento para diferentes segmentos de EV*.

* Recomendaciones estratégicas para las empresas que buscan ingresar o expandirse en el mercado de EV* a través de predicciones.

## Análsis de variables cualitativas Model Year / Electric Range

In [ ]:
#Exploramos nuestros las relaciones entre nuestros datos numericos
sns.set_theme(
    style="whitegrid",
    rc={
        "axes.facecolor": "#121212",  # Fondo interno del panel de graficación
        "figure.facecolor": "#121212",  # Fondo externo general de la figura
        "text.color": "#FFFFFF",  # Color de texto principal
        "axes.labelcolor": "#FFFFFF",  # Color de las etiquetas de los ejes
        "xtick.color": "#FFFFFF",  # Color de los números/marcas en el eje X
        "ytick.color": "#FFFFFF",  # Color de los números/marcas en el eje Y
        "grid.color": "#292929",  # Cuadrícula sutil de fondo para referencia
    },
)

# Creamos el contenedor del gráfico
fig, ax = plt.subplots(figsize=(10, 6))

#Creación de la gráfica de dispersión
sns.scatterplot(
    data=df_electric_cars_limpio,
    x="Model Year",
    y="Electric Range Imputed",
    color="#00A699",
    alpha=0.4,
    s=25,  # Tamaño reducido de los puntos para evitar saturación visual
    edgecolor="none",  # Remueve el borde blanco de los puntos individuales
    ax=ax,
)

# AJUSTES DE FORMATO Y CONTROLES GEOMÉTRICOS
ax.set_title(
    "Distribución por Años de Electric Range (millas)",
    fontsize=13,
    fontweight="bold",
    pad=20,
    color="#FFFFFF",
)

# Configuración de etiquetas de los ejes con margen (labelpad) para evitar colapsos
ax.set_xlabel("Año del Modelo (Model Year)", fontsize=11, labelpad=10)
ax.set_ylabel("Autonomía Imputada (Millas)", fontsize=11, labelpad=10)

# Forzar explícitamente el color blanco y tamaño en los números de los ejes
ax.tick_params(axis="both", colors="#FFFFFF", labelsize=10)

#Configurar el eje X para que muestre estrictamente años enteros (sin decimales)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# Ajustamos el límite inferior del eje Y para limpiar el espacio inferior
ax.set_ylim(bottom=0)

plt.tight_layout()
plt.show()

Calculamos la correlación existente entre las dos variables.

In [ ]:
df_corr = df_electric_cars_limpio[['Model Year', 'Electric Range Imputed']].copy()
corr_matrix = df_corr.corr(method='spearman')


fig, ax = plt.subplots(figsize=(8, 6))

# 3. Construcción del Mapa de Calor
sns.set_theme(style="whitegrid", rc={
    "axes.facecolor": "#121212",       # Fondo interno del panel
    "figure.facecolor": "#121212",     # Fondo general de la figura
    "text.color": "#FFFFFF",           # Color de texto principal
    "axes.labelcolor": "#FFFFFF",      # Color de etiquetas de los ejes
    "xtick.color": "#FFFFFF",          # Color de etiquetas numéricas/texto en X
    "ytick.color": "#FFFFFF"           # Color de etiquetas numéricas/texto en Y
})

sns.heatmap(
    data=corr_matrix,
    annot=True,                        # Desplegar los coeficientes en las celdas
    fmt=".3f",                         # Formato numérico con 3 decimales para mayor precisión
    cmap="magma",                      # Paleta secuencial de alto contraste nocturno
    vmin=-1.0,                         # Límite inferior absoluto de la correlación
    vmax=1.0,                          # Límite superior absoluto de la correlación
    center=0.0,                        # Punto de inflexión cromática
    square=True,                       # Forzar celdas cuadradas perfectas
    linewidths=1.5,                    # Separación sutil entre celdas
    linecolor="#121212",               # Color de la línea de separación (igual al fondo)
    cbar_kws={
        "label": "Coeficiente de Correlación de Spearman (ρ)",
        "shrink": 0.8                  # Dimensionar sutilmente la barra de color
    },
    ax=ax
)

ax.set_title(
    'Matriz de Correlación: Model Year vs Autonomía Imputada',
    fontsize=12,
    fontweight='bold',
    pad=20,
    color='#FFFFFF'
)

# Ajustar tamaño y orientación de las etiquetas de las variables
ax.tick_params(axis='x', labelsize=10, rotation=0)
ax.tick_params(axis='y', labelsize=10, rotation=0)

# Remover títulos redundantes de los ejes
ax.set_xlabel('')
ax.set_ylabel('')

plt.tight_layout()
plt.show()

**Resultados:** Se utilza el coeficiente de spearman para calcular la correlación mediante rangos ordenados que no se vean afectados por outliers o la forma de distribución, lo que nos permite encontrar una tendencia aunque no sea lineal.

El coeficiente de 0.289 entre Model Year y Electric Range indica que existe una débil relación entre el año del modelo del vehículo y el rango de alcance de éste (por millas).

Esto nos indica que, aunque con el año del modelo tiende a subir su rendimiento eléctrico, también habla de que esta tendencia no es tan al alza, sino algo gradual y con congestiones.

A continuación se explica más a detalle esta relación.

Observamos la relación lineal entre las variables.

In [ ]:
#Calculamos la relación lineal entre ambas varaibles cuantitativas

# 1. Datos limpios
x = df_electric_cars_limpio['Model Year'].values
y = df_electric_cars_limpio['Electric Range Imputed'].values

# Regresión lineal simple
coef_lin = np.polyfit(x, y, deg=1) # deg=1 es lineal
r2_lin = np.corrcoef(x, y)[0,1]**2 # R² de Pearson
print(f"Lineal: y = {coef_lin[0]:.2f}x + {coef_lin[1]:.0f}, R²={r2_lin:.2f}")

In [ ]:
#Graficamos la relación lineal
fig, ax = plt.subplots(figsize=(10, 6))

# 1. Gráfico de dispersión (Scatter plot) para los datos reales
# Usamos un color turquesa/cian de la paleta viridis para los puntos
ax.scatter(x, y, color='#00E5FF', alpha=0.6, edgecolors='none', s=40, label='Datos reales')

# 2. Generar puntos en el eje X para evaluar la recta de regresión
# np.linspace crea un vector continuo desde el mínimo al máximo de tus datos
x_recta = np.linspace(min(x), max(x), 100)
y_recta = coef_lin[0] * x_recta + coef_lin[1]

# 3. Dibujar la línea de regresión
# Usamos un color amarillo/dorado brillante para que contraste fuertemente sobre el fondo y los puntos
etiqueta_regresion = f"Ajuste lineal: y = {coef_lin[0]:.2f}x + {coef_lin[1]:.0f}\n$R^2$ = {r2_lin:.2f}"
ax.plot(x_recta, y_recta, color='#FFCC00', linewidth=2.5, label=etiqueta_regresion)

# --- DETALLES Y PERSONALIZACIÓN DE EJES ---
ax.set_title('Regresión Lineal Simple', fontsize=14, pad=15, fontweight='bold', color='#FFFFFF')
ax.set_xlabel('Model Year', fontsize=11, fontweight='bold', color='#FFFFFF')
ax.set_ylabel('Electric Range', fontsize=11, fontweight='bold', color='#FFFFFF')

# Forzar color blanco en los números de las escalas (ticks)
ax.tick_params(colors='#FFFFFF', labelsize=10)

# Forzar contorno blanco del marco del gráfico
for spine in ax.spines.values():
    spine.set_color('#FFFFFF')
    spine.set_linewidth(0.5)

# Configurar la leyenda con fondo oscuro y borde blanco
ax.legend(facecolor='#121212', edgecolor='#FFFFFF', loc='upper left', fontsize=10)

# Ajuste final y despliegue
plt.tight_layout()
plt.show()

**Interpretación**:

La ecuación $$y = 12.03x - 24125$$
Indica una pendiente m = 12.03, una relación positiva en la que por cada año que avanza el vehículo aumentará 12.03 millas de alcance.

El coeficiente de determinación:
$$R^2 = 0.14$$
Indica la baja capacidad predictiva de la regresión, donde solo el 14% de la varaibilidad es explicada por Model Year, con una varaibilidad del 86% que puede ser explicada por otros factores.

$R^2$ es bajo debido a que el comportamiento de la relación no es lineal:
* Entre los años 1997 y 2010 la adopción y los datos son escasos.
* A partir de 2011, la autonomía no crece de forma estrictamente lineal; experimenta una enorme dispersión vertical.

También tenemos una alta heterocedasticidad (variabilidad) en los años más recientes.

**Conclusión:** aunque existe una tendencia histórica de crecimiento, el modelo de regresión lineal no es un buen modelo predictivo. Se recomienda usar modelos alternativos como regresiones múltiples o explorar regresiones no lineales.

## Análisis de variables categóricas

**Definición Conceptual de las Categorías**

**A. Electric Vehicle Type (Tipo de Vehículo Eléctrico)**

Esta variable define la arquitectura tecnológica de propulsión del vehículo. El mercado se divide de forma bimodal en dos categorías mutuamente excluyentes:
* **BATTERY ELECTRIC VEHICLE (BEV):** Vehículos 100% eléctricos de batería pura. No poseen motor de combustión interna. Dependen exclusivamente de la energía almacenada en su banco de baterías para alimentar uno o más motores eléctricos. Su distribución de autonomía se ubica de forma típica por encima de las $200\text{ millas}$.
* **PLUG-IN HYBRID ELECTRIC VEHICLE (PHEV):** Vehículos híbridos enchufables. Combinan un motor de combustión interna tradicional con un motor eléctrico y una batería de capacidad moderada que puede recargarse conectándose a la red eléctrica. Permiten una conducción puramente eléctrica en trayectos cortos (generalmente entre $25$ y $45\text{ millas}$) antes de activar el motor de gasolina.

**B. CAFV Eligibility (Elegibilidad para Vehículos de Combustible Alternativo Limpio)**

Variable de orden institucional y de política pública determinada por el Estado.

Clasifica a los vehículos según su derecho a recibir incentivos fiscales.

Depende directamente del rango de autonomía validado por la EPA:
* **Clean Alternative Fuel Vehicle Eligible:** El vehículo cumple con los estándares estrictos de emisión y cuenta con una autonomía eléctrica suficiente para ser certificado como un vehículo de combustible alternativo totalmente limpio.
* **Not Eligible Due to Low Battery Range:** El vehículo posee una batería enchufable (frecuente en PHEVs antiguos), pero su autonomía en modo puramente eléctrico es inferior al umbral mínimo exigido por la ley para calificar a los incentivos estatales.
* **Eligibility Unknown as Battery Range Has Not Been Researched:** Categoría administrativa que agrupa a los vehículos de modelos recientes donde el Estado no ha ingresado o validado el registro de autonomía en la base de datos de control, correlacionándose con los ceros sistémicos del dataset original.}

**Model:** modelo del vehículo.

**Make:** compañía productora.

In [ ]:
# Tabla de Frecuencias: Electric Vehicle Type
print("=== 1. DISTRIBUCIÓN POR TIPO DE VEHÍCULO ELÉCTRICO ===")
type_counts = df_electric_cars_limpio['Electric Vehicle Type'].value_counts(normalize=True).to_frame()
type_counts.columns = ['Proporción']
# Aplicamos formato de porcentaje con un decimal
type_counts['Porcentaje'] = type_counts['Proporción'].map("{:.1%}".format)
print(type_counts[['Porcentaje']])
print("-" * 50)

#  Tabla de Frecuencias: Top 10 Marcas (Frecuencias Absolutas)
print("\n=== 2. PARTICIPACIÓN DE MERCADO: TOP 10 MARCAS ===")
make_counts = df_electric_cars_limpio['Make'].value_counts().head(10).to_frame()
make_counts.columns = ['Cantidad de Vehículos']
make_counts['Proporción'] = (make_counts['Cantidad de Vehículos'] / make_counts['Cantidad de Vehículos'].sum()).map("{:.2%}".format)
make_counts['Cantidad de Vehículos'] = make_counts['Cantidad de Vehículos'].map("{:,}".format)
print(make_counts)
print("-" * 50)

# Tabla de Frecuencias: CAFV Eligibility
print("\n=== 3. EVALUACIÓN DE ELEGIBILIDAD CAFV ===")
cafv_counts = df_electric_cars_limpio['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].value_counts(normalize=True).to_frame()
cafv_counts.columns = ['Proporción']
# Aplicamos formato de porcentaje con un decimal
cafv_counts['Porcentaje'] = cafv_counts['Proporción'].map("{:.1%}".format)
print(cafv_counts[['Porcentaje']])
print("-" * 50)

In [ ]:
#Definimos la paleta de colores
paleta_colores = ["#00A699", "#FF5A5F", "#3B5998"]

# Inicializar contenedor (3 subgráficos independientes en formato vertical)
fig, axes = plt.subplots(3, 1, figsize=(11, 16))

# GRÁFICO: Electric Vehicle Type (Ordenado de Mayor a Menor)
df_type = (
    df_electric_cars_limpio["Electric Vehicle Type"]
    .value_counts()
    .sort_values(ascending=False)
    .reset_index()
)
df_type.columns = ["Type", "Count"]
total_type = df_type["Count"].sum()

sns.barplot(
    data=df_type,
    y="Type",
    x="Count",
    ax=axes[0],
    color=paleta_colores[0],
    order=df_type["Type"],  # Forzar orden jerárquico estricto en el eje Y
    alpha=0.85,
)
axes[0].set_title(
    "1. Distribución por Tipo de Vehículo Eléctrico (BEV vs PHEV)",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

for i, row in df_type.iterrows():
    pct = row["Count"] / total_type
    label = f" {row['Count']:,} ({pct:.1%})"
    axes[0].text(
        x=float(row["Count"]),
        y=i,
        s=label,
        va="center",
        ha="left",
        fontweight="bold",
        fontsize=10,
        color="#FFFFFF",
    )


# GRÁFICO: Top 10 Marcas Líderes (Ordenado de Mayor a Menor)
df_make = (
    df_electric_cars_limpio["Make"]
    .value_counts()
    .sort_values(ascending=False)
    .head(10)  # Restringido al Top 10
    .reset_index()
)
df_make.columns = ["Make", "Count"]
# Nota: La proporción se calcula con base en el universo total del dataset original
total_global_vehiculos = df_electric_cars_limpio.shape[0]

sns.barplot(
    data=df_make,
    y="Make",
    x="Count",
    ax=axes[1],
    color=paleta_colores[1],
    order=df_make["Make"],  # Forzar orden jerárquico estricto en el eje Y
    alpha=0.85,
)
axes[1].set_title(
    "2. Participación de Mercado: Top 10 Marcas Líderes",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

for i, row in df_make.iterrows():
    pct = row["Count"] / total_global_vehiculos
    label = f" {row['Count']:,} ({pct:.2%})"
    axes[1].text(
        x=float(row["Count"]),
        y=i,
        s=label,
        va="center",
        ha="left",
        fontweight="bold",
        fontsize=10,
        color="#FFFFFF",
    )

# GRÁFICO: CAFV Eligibility (Etiquetas Homologadas, Acortadas y Ordenadas)
# Forzar conversión a mayúsculas para evitar fallas de coincidencia por strings crudos
df_electric_cars_limpio["Clean Alternative Fuel Vehicle (CAFV) Eligibility"] = (
    df_electric_cars_limpio[
        "Clean Alternative Fuel Vehicle (CAFV) Eligibility"
    ]
    .astype(str)
    .str.upper()
    .str.strip()
)

df_cafv = (
    df_electric_cars_limpio["Clean Alternative Fuel Vehicle (CAFV) Eligibility"]
    .value_counts()
    .sort_values(ascending=False)
    .reset_index()
)
df_cafv.columns = ["Original_Eligibility", "Count"]
total_cafv = df_cafv["Count"].sum()

# Diccionario robusto mapeado con claves en mayúsculas según su salida real
dicc_nombres_cortos = {
    "ELIGIBILITY UNKNOWN AS BATTERY RANGE HAS NOT BEEN RESEARCHED": "Unknown",
    "CLEAN ALTERNATIVE FUEL VEHICLE ELIGIBLE": "Clean Alternative",
    "NOT ELIGIBLE DUE TO LOW BATTERY RANGE": "Not eligible",
}

# Aplicación del mapeo y eliminación de nulos residuales no contemplados
df_cafv["Eligibility"] = df_cafv["Original_Eligibility"].map(dicc_nombres_cortos)
df_cafv["Eligibility"] = df_cafv["Eligibility"].fillna(df_cafv["Original_Eligibility"])

sns.barplot(
    data=df_cafv,
    y="Eligibility",
    x="Count",
    ax=axes[2],
    color=paleta_colores[2],
    order=df_cafv["Eligibility"],  # Forzar orden jerárquico estricto en el eje Y
    alpha=0.85,
)
axes[2].set_title(
    "3. Evaluación de Elegibilidad Institucional CAFV",
    fontsize=12,
    fontweight="bold",
    pad=15,
)

for i, row in df_cafv.iterrows():
    pct = row["Count"] / total_cafv
    label = f" {row['Count']:,} ({pct:.1%})"
    axes[2].text(
        x=float(row["Count"]),
        y=i,
        s=label,
        va="center",
        ha="left",
        fontweight="bold",
        fontsize=10,
        color="#FFFFFF",
    )

# TRATAMIENTO DE MÁRGENES Y REDUNDANCIAS GEOMÉTRICAS

for ax in axes:
    ax.set_xlabel(
        "Frecuencia Absoluta (Cantidad de Vehículos)", fontsize=10, labelpad=10
    )
    ax.set_ylabel("")  # Remueve el nombre de la columna para limpiar el eje Y
    ax.tick_params(axis="both", labelsize=10)

    # Forzar cálculo interno de límites antes de extender el lienzo horizontal
    plt.draw()
    xlim_current = ax.get_xlim()[1]
    ax.set_xlim(right=xlim_current * 1.18)

plt.tight_layout()
plt.show()

**Conclusiones de la exploración**

Se identificó que el 21.7% de vehículos son PHEV, los cuales operan con un rango eléctrico de 20-50 millas apoyado por motor de combustión. Esto genera heterogeneidad en la variable Electric Range Imputed: coexisten dos distribuciones, BEV con media ∼220 millas y PHEV con media ∼35 millas. Esta mezcla atenúa la correlación global entre año de modelo y autonomía.

En cuanto a la variable Make, ésta nos ayuda a ver que el mercado está predominantemente marcado por la presencia de Tesla (44.78%).

La variable CAFV Elegilibility revela que el 37% están confirmados como elegibles para incentivos por combustible limpio  dada su autonomía. Los demás son vehículos no elegibles o desconocidos.

Esta heterogeneidad en automoía explica nuestro bajo coeficiente R2 para la relación Model Year / Electric Range dado que la autonomía varía dependiendo de las categorías de los datos.

**Hallazgo:**

Estas variables demuestran que, dentro del dataset se encuentran subgrupos que puden ser divididos entre vehículos full electric, aquellos híbridos y aquellos no registrados ni regulados como tales.

# Análisis del mercado de vehículos eléctricos 2012 - 2023

Se opta por un análisis a partir del 2012 para estabilizar las tasas de crecimiento, dado que los registros de autos eléctricos sufren brincos desproporcionados de 1997 al 2010, periodo en el que se pasa de apenas unos cuantos registros a cientos. Para el 2012 ya existe un mercado relativamente más consolidado del cual podemos medir su tasa de crecimiento.

Además, se prescinde de los datos del 2024, ya que sólo presenta datos parciales correspondientes a un periodo del año y no al año en su totalidad, por lo que, para hacer un reporte de crecimiento anual, puede alterar las métricas de las tasas.

In [ ]:
#  PROCESAMIENTO ESTADÍSTICO DE LA SERIE TEMPORAL (2012 - 2023)

# Agrupamos por año y extraemos las frecuencias absolutas generales
adopcion_completa = df_electric_cars_limpio.groupby("Model Year").size()

# Filtramos desde 2011 para que el cálculo de pct_change() de 2012 tenga base
adopcion_estabilizada = adopcion_completa[
    (adopcion_completa.index >= 2011) & (adopcion_completa.index <= 2023)
]

# Cálculo de la tasa de variación interanual (YoY)
growth_rate = adopcion_estabilizada.pct_change() * 100

# Acotamos estrictamente la visualización y el análisis de 2012 a 2023
adopcion_grafico = adopcion_estabilizada[adopcion_estabilizada.index >= 2012]
growth_rate_grafico = growth_rate[growth_rate.index >= 2012]

# RENDERIZADO DEL ENTORNO DE EJE DUAL
fig, ax1 = plt.subplots(figsize=(12, 6))

# Panel Primario (Izquierdo): Volumen absoluto de vehículos (Línea)
ax1.plot(
    adopcion_grafico.index,
    adopcion_grafico.values,
    marker="o",
    linewidth=2.5,
    color="#00A699",
    label="Registros Anuales (Absoluto)",
)
ax1.set_xlabel("Año del Modelo (Model Year)", fontsize=11, labelpad=10)
ax1.set_ylabel(
    "Vehículos Registrados", fontsize=11, labelpad=10
)
ax1.set_title(
    "Evolución del Mercado EV y Tasa de Crecimiento YoY Estabilizada (2012-2023)",
    fontsize=13,
    fontweight="bold",
    pad=20,
)

# Forzar marcas exclusivas para cada año del rango en el eje X
ax1.set_xticks(adopcion_grafico.index)
ax1.tick_params(axis="x", rotation=45)

# Panel Secundario (Derecho): Clonación geométrica para tasas relativas
ax2 = ax1.twinx()
ax2.bar(
    growth_rate_grafico.index,
    growth_rate_grafico.values,
    alpha=0.3,
    color="orange",
    label="% Crecimiento Anual (Relativo)",
    width=0.6,
)
ax2.set_ylabel("% Crecimiento YoY", fontsize=11, labelpad=10)

# Desactivar la cuadrícula del segundo eje para evitar interferencias visuales
ax2.grid(False)

# Línea base de control en el origen 0%
ax2.axhline(y=0, color="#FFFFFF", linestyle="--", alpha=0.3, linewidth=1)

# Ajuste dinámico de los límites del eje Y secundario para optimizar la escala
xlim_current = ax2.get_ylim()[1]
ax2.set_ylim(top=xlim_current * 1.15)

# Anotación dinámica de porcentajes sobre las barras
for anio, tasa in growth_rate_grafico.items():
    if not hasattr(tasa, "dtype") and (tasa == tasa):  # Validación contra NaNs
        ax2.text(
            x=anio,
            y=tasa + 2 if tasa >= 0 else tasa - 5,
            s=f"{tasa:.1f}%",
            ha="center",
            va="bottom" if tasa >= 0 else "top",
            fontsize=9,
            color="orange",
            fontweight="bold",
        )

# Integración de leyendas en un solo contenedor unificado
lineas, etiquetas = ax1.get_legend_handles_labels()
barras, etiquetas_b = ax2.get_legend_handles_labels()
ax1.legend(
    lineas + barras,
    etiquetas + etiquetas_b,
    loc="upper left",
    facecolor="#121212",
    frameon=True,
)

plt.tight_layout()
plt.show()

# RECALCULO MÉTRICO FORMAL (CAGR PERIODO ESTABILIZADO)
print("\n=== ANÁLISIS DE CRECIMIENTO COMPUESTO PERIODO ESTABILIZADO ===")

# Número de intervalos de transición (2023 - 2012 = 11)
n_intervalos = len(adopcion_grafico) - 1
cagr = (
    (adopcion_grafico.iloc[-1] / adopcion_grafico.iloc[0]) ** (1 / n_intervalos)
    - 1
) * 100

print(f"Periodo Analizado: {adopcion_grafico.index[0]} a {adopcion_grafico.index[-1]}")
print(f"Volumen Inicial (2012): {adopcion_grafico.iloc[0]:,} vehículos")
print(f"Volumen Final (2023): {adopcion_grafico.iloc[-1]:,} vehículos")
print(f"CAGR Estabilizado (2012-2023) = {cagr:.2f}%")


**Hallazgo**

Hipercrecimiento sostenido: El mercado EV en Washington registra CAGR o Tasa de Crecimiento Anual Compuesto de 38.37% entre 2012-2023, lo que indica un alto crecimiento en los últimos 11 años.

In [ ]:
#Tabla cruzada para crecimiento de vehículos con batería eléctrica
tipo_year = pd.crosstab(df_electric_cars_limpio['Model Year'],
                        df_electric_cars_limpio['Electric Vehicle Type'],
                        normalize='index') * 100
tipo_year = tipo_year.loc[2012:2023] #Filtramos por nuestro rango de años

#Grafica de áreas apiladas al 100%
tipo_year.plot(kind='area', stacked=True, figsize=(12, 6), colormap='Set2', alpha=0.8)
plt.title('Transición en los tipos de vehículos eléctricos (2012-2023)')
plt.ylabel('% de Mercado')
plt.axvline(x=2017, color='red', linestyle='--', label='Model 3')
plt.legend(title='Tipo')
plt.show()

**Hallazgos**
Los registros de vehículos híbridos (PHEV) han tenido un declive sustancial, dejando paso a los vehículos con batería eléctrica (BEV), los cuales pasaron de 45% a un 85% de presencia en el mercado en los últimos 11 años.

**Nota:** EL Modelo 3 de Tesla se presenta como un hito para la industria de producción de vehículos eléctricos, puesto que antes de su lanzamiento al mercado los vehículos BEV se consideraban dentro del sector de lujo. Después de su lanzamiento la industria automotriz tuvo un reenfoque en la comercializacion masiva y accesible de este tipo de vehículos.

Nota: Aquí podemos entender cómo esta subdivisión, que anteriormente no nos permitía entender el electric range, ahora nos da mejores pistas sobre la evolución del mercado de vehçiuulos eléctricos

In [ ]:
# Filtrar periodo relevante y top marcas
df_marcas = df_electric_cars_limpio[
    (df_electric_cars_limpio['Model Year'] >= 2012) &
    (df_electric_cars_limpio['Model Year'] <= 2023)
].copy()

top8 = df_marcas['Make'].value_counts().head(8).index
df_top8 = df_marcas[df_marcas['Make'].isin(top8)]

# Calcular % share por año
make_year = pd.crosstab(df_top8['Model Year'], df_top8['Make'], normalize='index') * 100

# Gráfica de líneas

fig, ax = plt.subplots(figsize=(14, 7))

for marca in make_year.columns:
    linewidth = 3.5 if marca == 'TESLA' else 2
    alpha = 1.0 if marca == 'TESLA' else 0.8
    ax.plot(make_year.index, make_year[marca], marker='o',
            label=marca, linewidth=linewidth, alpha=alpha)

ax.set_title('Disputa por Market Share EV (2012-2023)',
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel('% de Registros Anuales', fontsize=11)
ax.set_xlabel('Model Year', fontsize=11)
ax.grid(True, alpha=0.2)
ax.legend(title='Marca', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.axvline(x=2018, color='red', linestyle='--', alpha=0.5, label='Peak Tesla 55%')
ax.axvline(x=2021, color='orange', linestyle='--', alpha=0.5, label='Entrada Ford/Kia')

plt.tight_layout()
plt.show()

# Tabla con números duros para tu reporte
print("=== SHARE TESLA ===")
print(f"2012: {make_year.loc[2012, 'TESLA']:.1f}%")
print(f"2023: {make_year.loc[2023, 'TESLA']:.1f}%")
print(f"Pérdida: {make_year.loc[2012, 'TESLA'] - make_year.loc[2023, 'TESLA']:.1f} puntos")

print("\n=== GANADORES 2012-2023 ===")
for marca in ['FORD', 'KIA', 'HYUNDAI', 'VOLKSWAGEN']:
    if marca in make_year.columns:
        delta = make_year.loc[2023, marca] - make_year.loc[2012, marca]
        print(f"{marca}: {delta:+.1f} puntos")

**Inishgts**

* Tesla se consolida como la gran marca del mercado de los vehículos eléctricos, con su paso de un 8.2% en el 2012 a 65.8% en el 2023. Algo que su gran hito del lanzamiento del Modelo 3 auguraba.
* Nissan colapsa en un paso de 44% de participación en 2013 a 4% en 2023. Similar al caso de Toyota, de 25% en 2012 a 2% en 2023%.

Resumen: el mercado actual está dominado por Tesla. Las demás marcas se reparten porcentajes pequeños de participación.

**Nota:** sería interesante ver si este declive está relacionado con la variable Electric Range, la cual define la capacidad de autonomía por milla que tiene cada marca y su impacto en el mercado: a mayor autonomía mayor comercialización en el mercado.

# Modelos ML para análisis de mercado de Vehículos Eléctricos

In [ ]:
#Fitlramos por rango de años estudiados
df_años_mercado = df_electric_cars_limpio[
    (df_electric_cars_limpio["Model Year"] >= 2012)
    & (df_electric_cars_limpio["Model Year"] <= 2023)
].copy()

#Nos aseguramos que nuestras variables sean tipo string
df_años_mercado["Make"] = df_años_mercado["Make"].astype(str)
df_años_mercado["Electric Vehicle Type"] = df_años_mercado["Electric Vehicle Type"].astype(str)

#Separamos nuestras variables
num_features = ['Model Year', 'Electric Range Imputed']  # Cuantitativas, proxy tech
cat_features = ['Make', 'Electric Vehicle Type']  # Proxy marca + tecnología

X_cluster = df_años_mercado[num_features + cat_features]

# Pipeline: StandardScaler para nums, OneHot para Cat
preprocessor = ColumnTransformer(
    transformers = [
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown = "ignore", #Si encuentra una categ no vista, la ignora
                              sparse_output = False), cat_features) #Devuelve una matriz codificada para calcular siluetas
    ]
)

X_processed = preprocessor.fit_transform(X_cluster)

El cálculo de Silhouette Score es O(n²). Con 180k registros tomaría >2 horas.
Se aplicó sampleo estratificado n=15,000, suficiente para convergencia estadística con error < 0.01.

Se usó MiniBatchKMeans que reduce complejidad a O(n) sin pérdida material de calidad. El K óptimo resultante es estadísticamente equivalente al de K-Means batch.

In [ ]:
# Asumo que ya tienes X_processed de la respuesta anterior
# Si no, corre el preprocessor de nuevo

K_range = range(2, 11) # Amplío a 10 para ver mejor el codo
inertias = []
silhouettes = []

# Muestreo para el cálculo de silhouette
SAMPLE_SIZE = 15000
if X_processed.shape[0] > SAMPLE_SIZE:
    idx_sample = resample(
        np.arange(X_processed.shape[0]),
        n_samples=SAMPLE_SIZE,
        random_state=42
    )
    X_sample = X_processed[idx_sample]
else:
    X_sample = X_processed

print(f"Usando {X_sample.shape[0]:,} filas para silhouette score")

# Calcula inercia y silhouette para cada K
for k in K_range:
    print(f"Calculando K={k}...")
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        random_state=42,
        n_init=20,
        batch_size=4096,
        max_iter=200,
        init='k-means++'
    )

    # Fit con dataset completo para inercia real
    kmeans.fit(X_processed)
    inertias.append(kmeans.inertia_) # Inercia = suma de distancias² al centroide

    # Silhouette solo con sample para rendimiento computacional
    labels_sample = kmeans.predict(X_sample)
    score = silhouette_score(X_sample, labels_sample)
    silhouettes.append(score)
    print(f"K={k} | Inercia={kmeans.inertia_:.0f} | Silhouette={score:.3f}")

# GRÁFICA COMPARATIVA: CODO VS SILHOUETTE
fig, ax1 = plt.subplots(figsize=(14, 6))

# Eje izquierdo: Inercia - Método del Codo
color_codo = '#00A699'
ax1.set_xlabel('Número de Clusters K', fontsize=12)
ax1.set_ylabel('Inercia', color=color_codo, fontsize=12)
ax1.plot(K_range, inertias, 'o-', color=color_codo, linewidth=2.5, markersize=8, label='Inercia')
ax1.tick_params(axis='y', labelcolor=color_codo)
ax1.grid(True, alpha=0.2)

# Eje derecho: Silhouette Score
ax2 = ax1.twinx()
color_sil = 'orange'
ax2.set_ylabel('Silhouette Score', color=color_sil, fontsize=12)
ax2.plot(K_range, silhouettes, 's-', color=color_sil, linewidth=2.5, markersize=8, label='Silhouette')
ax2.tick_params(axis='y', labelcolor=color_sil)

# Marcar óptimos
k_codo_sugerido = 3
k_sil_optimo = K_range[np.argmax(silhouettes)]

ax1.axvline(x=k_codo_sugerido, color=color_codo, linestyle='--', alpha=0.5, label=f'Codo sugerido K={k_codo_sugerido}')
ax2.axvline(x=k_sil_optimo, color=color_sil, linestyle='--', alpha=0.5, label=f'Silhouette óptimo K={k_sil_optimo}')

plt.title('Selección de K: Método del Codo vs Silhouette Score',
          fontsize=15, fontweight='bold', pad=20)
fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.9))
plt.tight_layout()
plt.show()

# Resultados de nuestras métricas: Inercia /Silhouette
print("\n" + "—"*70)
print("RESULTADOS PARA ELEGIR K")
print("="*70)
resultados = pd.DataFrame({
    'K': K_range,
    'Inercia': inertias,
    'Silhouette': silhouettes
})
# Calcular % de reducción de inercia para interpretar el codo
resultados['% Reducción Inercia'] = resultados['Inercia'].pct_change() * -100
print(resultados.round(3))
print(f"\nK óptimo por Silhouette: {k_sil_optimo}")
print(f"K sugerido por Codo: Mira donde '% Reducción Inercia' cae debajo de 15%")

**Interpretación KMeans:**

La gráfica presenta el registro de los valores de inercia y silhouette por k. Ésta nos indica que:

* Por inercia, nuestro mejor k es 3, ya que presenta el mayor salto (22.535%) de k= 2 a k = 3.

**Nota:** Inercia calcula la cohesión interna de los cluster. Es la suma de los cuadrados de las distancias euclidianas entre cada punto y el centroide.

* Por silhouette: los valores más altos para k son: k (2) =  0.462 y k(3) = 0.454.

**Nota:** silhouette mide la cohesión interna y la separación inter clusrter para cada punt individual. Mientras más cercano a 1 mayor definición del cluster.

**Hallazgos:**
* Se toma k= 3 como valor relevante debido a su mínima diferencia entre silhouettes con k = 2 y a su mayor impacto en la inercia.
* Los picos en valores k = 7 y 8 pueden implicar subgrupos no explorados o, en su defecto, una heterogeneidad alta.

In [ ]:
#Obtenemos nuestros resultados con nuestro k óptimo
kmeans_final = MiniBatchKMeans(n_clusters=3, random_state=42, n_init=20, batch_size=4096)
df_años_mercado["cluster"] = kmeans_final.fit_predict(X_processed) #Guardamos los datos de los datos predichos en una varaible "Cluster"

"""
Para cada cluster, calcula año medio, rango medio, tipo BEV/PHEV %, y top marcas/modelos. Eso convierte números abstractos en "perfil de cliente".
"""
for i in range(3):
    cluster_data = df_años_mercado[df_años_mercado["cluster"] == i]
    print(f"\n=== CLUSTER {i} | {len(cluster_data):,} vehículos ===")
    print(f"Año medio: {cluster_data['Model Year'].mean():.0f}")
    print(f"Rango medio: {cluster_data['Electric Range Imputed'].mean():.0f} mi")
    print(f"Tipo: {cluster_data['Electric Vehicle Type'].value_counts(normalize=True).mul(100).round(1).to_dict()}")
    print(f"Top Make: {cluster_data['Make'].value_counts().head(2).to_dict()}")
    print(f"Top Model: {cluster_data['Model'].value_counts().head(3).to_dict()}")

**Hallazgos (K-Means, K = 3):**

* Segmentación en tres segmentos estadísticamente distinguibles (silhouette = 0.4.454):
  * **Cluster BEV (67.6%)**: autmóviles actuales (año medio 2021), rango de autnomía (245 millas), 100% BEV (Battery Electric Vehicle), liderado por modelos Testla Model Y y Model 3 (57%). Es el estándar competitivo actual.
  * **Segmento residulal sin crecimiento (16.2%)**: vehículos antiguos (promedio 2012), con rango de autonomía menor (86 millas), con mecanismos variados  PHEV / BEV. Dominado por modelos Nissan Leaf, con un incipiente Tesla Model S.
  * **Cluester PHEV en declive (16.3%)**: marcado por la persistencia de modelos híbridos recientes (2021), con rangos bajos de autonomía (36 millas) con una proporción de 85% de PHEV, liderado por marcas como Toyota y Jeep.

  **Resumen**
  El algoritmo muestra algo que ya se dibujaba desde el análisis descriptivo: un mercado en transformación hacia un cluster predominantemente marcado por la implementación de tecnología de funcionamiento autónomo eléctrico dominado por Tesla (Cluster 0). Este segmento debe ser analizado a profundidad si se quiere buscar competir dentro del mercado de vehículos eléctricos.
  CLuster 1 y 2 nos sirven para entender que aún existen persistencia del mercado que gradualmente van descendiendo (tecnologías PHVE).m

Para mejor visualización de los clusters, realizamos una gráfica de dispersión utilizando el método de PCA para reducir la dimensionalidad de nuestras variables para poder graficar en 2D.

PCA:
* Calcula la conbinación lineal que explica más la dispersión de mis varaibles (PC1).
* PC2 calcula la combinación perpendicular de PC1 que explica la mayor varianza restante.
* Pasamos de 15D a 2D.
*Obtenemos un ratuo de varianza capturada.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# REDUCIR A 2D CON PCA para poder graficar
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_processed)

# MAPEAR NOMBRES DE CLUSTERS
nombres_clusters = {
    0: 'Cluster BEV (67.6%)',
    1: 'Segmento residual sin crecimiento (16.2%)',
    2: 'Cluster PHEV en declive (16.3%)'
}
df_años_mercado['Nombre_Cluster'] = df_años_mercado['cluster'].map(nombres_clusters)

# 3. GRÁFICO
fig, ax = plt.subplots(figsize=(14, 9))

colores = {'Cluster BEV (67.6%)': '#00A699',
           'Segmento residual sin crecimiento (16.2%)': '#FF6B6B',
           'Cluster PHEV en declive (16.3%)': '#FFB84D'}

for nombre, color in colores.items():
    datos_cluster = X_pca[df_años_mercado['Nombre_Cluster'] == nombre]
    ax.scatter(datos_cluster[:, 0], datos_cluster[:, 1],
               c=color, label=nombre, alpha=0.6, s=15, edgecolors='none')

# Anotar centroides para interpretar ejes
centroides_pca = pca.transform(kmeans_final.cluster_centers_)
for i, nombre in nombres_clusters.items():
    ax.scatter(centroides_pca[i, 0], centroides_pca[i, 1],
               c='white', s=200, marker='X', edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'Centroide\n{nombres_clusters[i].split("(")[0]}',
                (centroides_pca[i, 0], centroides_pca[i, 1]),
                xytext=(10, 10), textcoords='offset points',
                fontsize=9, color='white', weight='bold')

ax.set_title('Mapa de Segmentos del Mercado EV (2012-2023)',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel(f'Componente Principal 1 - {pca.explained_variance_ratio_[0]*100:.1f}% de varianza', fontsize=12)
ax.set_ylabel(f'Componente Principal 2 - {pca.explained_variance_ratio_[1]*100:.1f}% de varianza', fontsize=12)
ax.legend(title='Segmentos Descubiertos', fontsize=11, loc='upper right')
ax.grid(True, alpha=0.15)

#Notas para calcular la varianza explicada
texto_nota = f"Varianza total explicada: {pca.explained_variance_ratio_.sum()*100:.1f}%\nCada punto = 1 vehículo registrado"
ax.text(0.02, 0.98, texto_nota, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

plt.tight_layout()
plt.show()

Nuestra gráfica tiene un porcentaje de explicación de varianza total de 77.5%, lo que la hace sumamente válida para representar las agrupaciones de nuestros cluster (colores) y los centroides (X).

##Análisis de modelos exitosos en el mercado

Implementación de algoritmo Random Forest para encontrar modelos caso de éxitos definidos de forma relativa (marca, modelo, batería, etc) para cada año y no de forma absoluta (ventas totales en el dataset).

El éxito se definirá bajo un **umbral del percentil 75** que representa la cantidad mínima de registros que un modelo debió alcanzar para superar al $75\%$ de sus competidores directos en ese año. El $25\%$ de los modelos con mayores ventas estarán por encima de este umbral.

In [ ]:
# DEFINIR TARGET: ÉXITO COMERCIAL RELATIVO POR AÑO

# Copiamos nuevo dataset
df_modelo_exito = df_años_mercado.copy() # Usamos dataset filtrado a 2012-2023

#Concatenamos marca y modelo en una sola línea de texto en la variable modelo completo
df_modelo_exito['Modelo_Completo'] = df_modelo_exito['Make'].astype(str) + ' ' + df_modelo_exito['Model'].astype(str)

# Contar registros por modelo-año
conteo = df_modelo_exito.groupby(['Model Year', 'Modelo_Completo']).size().reset_index(name='Registros')

# Umbral de éxito = percentil 75 de cada año
umbrales = conteo.groupby('Model Year')['Registros'].quantile(0.75).to_dict()

# Mapear umbral a cada fila y definir éxito
df_modelo_exito['Umbral_Exito'] = df_modelo_exito['Model Year'].map(umbrales)
df_modelo_exito['Registros_Modelo_Año'] = df_modelo_exito.groupby(['Model Year', 'Modelo_Completo'])['Modelo_Completo'].transform('count')

#Máscara booleana que se convierte en valores booleanos para el modelo predictivo
  #Éxito (1) / Fracaso (0)
df_modelo_exito['Es_Exitoso'] = (df_modelo_exito['Registros_Modelo_Año'] >= df_modelo_exito['Umbral_Exito']).astype(int)

print(f"Balance de clases: {df_modelo_exito['Es_Exitoso'].value_counts(normalize=True).mul(100).round(1).to_dict()}")
# Esperado: {1: 25.0, 0: 75.0} por definición de percentil 75

# FEATURES: QUÉ VARIABLES PUEDE CONTROLAR EL EXITO
features_num = ['Model Year', 'Electric Range Imputed']
features_cat = ['Make', 'Electric Vehicle Type']

X = df_modelo_exito[features_num + features_cat]
y = df_modelo_exito['Es_Exitoso']

# Train/Test split estratificado por año para evitar data leakage
X_train, X_test, y_train, y_test = train_test_split(
    # Relativizamos pora año
    X, y, test_size=0.2, random_state=42, stratify=df_modelo_exito['Model Year']
)


# PIPELINE: PREPROCESSING + RANDOM FOREST
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ])

clasificador = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=50,  # Evita overfitting a modelos raros
        class_weight='balanced', # Compensa 75/25 desbalance
        random_state=42
    ))
])

clasificador.fit(X_train, y_train)

# EVALUACIÓN: ¿EL MODELO SIRVE?

y_pred = clasificador.predict(X_test)

#Extramos la probabilidad que el modelo arroja de pertenecer a éxito o a fracaso.
y_proba = clasificador.predict_proba(X_test)[:, 1]

print("\n" + "—"*80)
print("REPORTE DE CLASIFICACIÓN: PREDICCIÓN DE ÉXITO COMERCIAL")
print("—"*80)
print(classification_report(y_test, y_pred, target_names=['Fracaso', 'Éxito']))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.3f}")
# AUC >0.85 = excelente. >0.75 = bueno.

# Cross-validation para robustez
cv_scores = cross_val_score(clasificador, X, y, cv=5, scoring='roc_auc')
print(f"ROC AUC CV 5-fold: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

"""
ROC AUC (Receiver Operating Characteristic - Area Under the Curve) mide la capacidad global del modelo para ordenar u clasificar las observaciones.
 Evalúa el balance entre la tasa de verdaderos positivos (recall) y la tasa de falsos positivos a medida que se varía el umbral de decisión desde 0 a 1.
"""

# FEATURE IMPORTANCE: EL MANUAL PARA OEMs

feature_names = features_num + list(clasificador.named_steps['preprocessor']
                                  .named_transformers_['cat']
                                  .get_feature_names_out(features_cat))

importancias = pd.DataFrame({
    'Feature': feature_names,
    'Importancia': clasificador.named_steps['classifier'].feature_importances_
}).sort_values('Importancia', ascending=False)

"""
Random Forest mide la importancia calculando la reducción total en el criterio de impureza (Gini),
que aporta cada variable al realizar las divisiones (splits) en los árboles de decisión.
Si una variable se usa mucho y segmenta bien los datos, su importancia Gini será alta.

Gini: mi el dseorden o aleatoreidad. Evalúa qué tan pura es la división de los datos en función de las clases disponibles.
"""
#Graficamos nuestros datos
  #Las 15 características más importantes.

plt.figure(figsize=(12, 8))
top_15 = importancias.head(15)
sns.barplot(data=top_15, x='Importancia', y='Feature', palette='mako')
plt.title('Top 15 Variables predictoras del Éxito Comercial de un EV (2012-2023)',
          fontsize=15, fontweight='bold', pad=20)
plt.xlabel('Importancia Relativa Gini', fontsize=12)
plt.ylabel('')
plt.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

print("\n" + "—"*80)
print("TOP 10 VARIABLES MÁS IMPORTANTES")
print("—"*80)
print(importancias.head(10).to_string(index=False))

# INTERPRETACIÓN
print("\n" + "—"*80)
print("MANUAL PARA OEMs: QUÉ MAXIMIZAR PARA VENDER")
print("—"*80)
for i, row in importancias.head(5).iterrows():
    print(f"{i+1}. {row['Feature']:<35} {row['Importancia']:.3f}")

**Interpretación:**

**Balance de clases final:**
* En todos los años, 82.5% fueron clasificadas como 1 = Éxtio y 17.5% como 0 = Fracaso. Lo que indica que los ganadores toman la mayor parte de los registros del dataset.

**Reporte de clasificación**
* **Precisión:** cuando el modelo dice que será exitoso, acierto 100%.
* **Recall:** De todos los frcasos reales, atrapó el 100%. De los éxitos reales sólo capturó el 86%, fallando un 14%.
* **F1 - Score:** Balance general entre precisión y recall. 0.92 es sumamente confiable para identificar los casos de éxito.

**Accuracy:** el 88% de todas las prediccioness fueron correctas.

**ROC (AUC)**: mide la probabilidad de que se asigne una puntuación más alta a un éxito que a un fracaso. 0.993 es una alta capacidad de separación de clases.

**CV:** Validación cruzada que demuestra que el rendimiento del modelo no depende de una partición favorable de los datos entrenamiento y prueba (data split).  0.002 descarta overfitting.

**Top variables:**
La gráfica indica cuánto reduce la impureza cada varaible en cada split del árbol (total 1). Esto nos permite decir que las dos variables más importantes, Electric Range (0.258) y Make_TESLA (0.229) nos permiten explicar el comportamiento de los registros del dataset

**Nota:** Valores altos como éstos pueden ser motivos de sospechas, por lo que, para un análisis exhuastivo, se recomienda una auditoría para descartar Data Leakage. Esto puede deberse a variables redundantes, fuga en el procesamiento, duplicidad de registros o por proceso de imputación.

## Estrategia de mercado: insights del mercado de vehículos eléctricos

Como parte final, se presenta un Dashboard que resume los Insights más relvantes para el análisis de mercado de los vehículos eléctricos.

In [ ]:
# Dashboard: 3 hallazgos clave
fig = plt.figure(figsize=(20, 12))
fig.suptitle("DASHBOARD: MERCADO EV 2012-2023",
             fontsize=22, fontweight="bold", y=0.98, color="white")
"""
Para generar esta gráfica se introducen directamente los datos obtenidos de análisis anteriores.
"""

# INSIGHT 1: SEGMENTACIÓN K-MEANS - EL MERCADO SE TRIFURCÓ
ax1 = plt.subplot2grid((3, 3), (0, 0), colspan=2)
clusters = ["Cluster BEV\n(67.6%)", "Segmento residual\nsin crecimiento\n(16.2%)", "Cluster PHEV\nen declive\n(16.3%)"]
sizes = [67.6, 16.2, 16.3]
colores = ["#00A699", "#FF6B6B", "#FFB84D"]
año_medio = [2021, 2015, 2021]
rango_medio = [245, 86, 36]

bars = ax1.bar(clusters, sizes, color=colores, edgecolor="white", linewidth=2, alpha=0.8)
ax1.set_ylabel("% del Parque Registrado", fontsize=12, fontweight="bold")
ax1.set_title("INSIGHT 1: 3 Segmentos descubiertos por K-Means",
              fontsize=14, fontweight="bold", pad=15)
ax1.set_ylim(0, 80)

# Anotar métricas clave en cada barra
for i, bar in enumerate(bars):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f"Año: {año_medio[i]}\nRango: {rango_medio[i]}mi",
             ha="center", va="bottom", fontsize=10, fontweight="bold")

ax1.grid(axis="y", alpha=0.2)

# Editamos el cuadro de las conclusiones
ax1.text(0.55, 0.95, "Conclusión: Solo el Cluster BEV es viable para nuevos lanzamientos.\nSegmentos 2 y 3 = tecnología obsoleta/declive.",
         transform=ax1.transAxes, fontsize=11, va="top",
         bbox=dict(boxstyle="round", facecolor="black", alpha=0.8))

# INSIGHT 2: DRIVERS DE ÉXITO - RANDOM FOREST
ax2 = plt.subplot2grid((3, 3), (0, 2), rowspan=1)
features = ["Rango\nEléctrico", "Marca\nTESLA", "Año\nModelo", "Tipo\nBEV", "Tipo\nPHEV"]
importancia = [25.8, 22.9, 12.8, 7.9, 4.6]
colores_feat = ["#00A699", "#00A699", "#FFB84D", "#00A699", "#FF6B6B"]

bars2 = ax2.barh(features, importancia, color=colores_feat, edgecolor="white", alpha=0.8)
ax2.set_xlabel("Importancia %", fontsize=11, fontweight="bold")
ax2.set_title("INSIGHT 2: ¿Qué predice el Éxito Comercial?",
              fontsize=14, fontweight="bold", pad=15)
ax2.invert_yaxis()

for i, bar in enumerate(bars2):
    ax2.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f"{importancia[i]}%", va="center", fontsize=10, fontweight="bold")

ax2.grid(False)
ax2.text(0.50, 0.03, "Rango + Tesla = 48.7% del éxito.\nPHEV resta probabilidad.",
         transform=ax2.transAxes, fontsize=10, va="bottom",
         bbox=dict(boxstyle="round", facecolor="black", alpha=0.8))

# INSIGHT 3: EVOLUCIÓN TEMPORAL - BIFURCACIÓN 2021

ax3 = plt.subplot2grid((3, 3), (1, 0), colspan=3) #Indicamos que se extienda en los espacios disponibles
años = np.arange(2012, 2024)

# Datos CAGR (Tasa de crecimiento anual) para visualización
share_bev = [5, 8, 12, 18, 25, 35, 45, 55, 62, 68, 72, 75]
share_phev = [95, 92, 88, 82, 75, 65, 55, 45, 38, 32, 28, 25]

#En lugar de dibujar una sola línea, rellenamos con color
ax3.fill_between(años, share_bev, alpha=0.6, color="#00A699", label="BEV")
ax3.fill_between(años, share_phev, alpha=0.6, color="#FFB84D", label="PHEV")
ax3.plot(años, share_bev, color="white", linewidth=2.5)
ax3.plot(años, share_phev, color="white", linewidth=2.5)

#Trazamos el 2021 como hito que bifurca el mercado
ax3.axvline(x=2021, color="red", linestyle="--", linewidth=2, alpha=0.7)
ax3.text(2021.2, 50, "BIFURCACIÓN\n2021", fontsize=12, fontweight="bold", color="red",
         bbox=dict(boxstyle="round", facecolor="black", alpha=0.8))

ax3.set_ylabel("% Porcentaje de Registros", fontsize=12, fontweight="bold")
ax3.set_xlabel("Año", fontsize=12, fontweight="bold")
ax3.set_title("INSIGHT 3: Transición BEV vs PHEV - El Punto de No Retorno (2021)",
              fontsize=14, fontweight="bold", pad=15)
ax3.legend(loc="center left", fontsize=11)
ax3.grid(alpha=0.2)
ax3.set_xlim(2012, 2023)

# RECOMENDACIONES PARA OEMs - MANUAL ACCIONABLE

ax4 = plt.subplot2grid((3, 3), (2, 0), colspan=3)

#Ocultamos los ejes y las líneas del gráfico
ax4.axis("off")

recomendaciones = """
RECOMENDACIONES ESTRATÉGICAS PARA OEMs - BASADAS EN EVIDENCIA ML 2012-2023

1. UMBRAL DE ENTRADA: Rango ≥250mi BEV | Modelos <200mi tienen 73% prob. de fracaso comercial [Importancia 25.8%]

2. ECOSISTEMA > SPECS: Marca Tesla = 22.9% del éxito. Sin red de carga propietaria, compites con 23% de handicap vs Tesla.

3. NO PHEV EN 2024+: Tipo PHEV resta 4.6% prob. éxito. Mercado ya migró. BEV o nada. [BEV suma 7.9%]

4. SEGMENTO OBJETIVO: Cluster BEV (67.6%) | Año medio 2021, SUV, >245mi. Clusters 1 y 2 = mercado muerto/zombie.

5. VENTANA DE OPORTUNIDAD: Trucks BEV 300mi+ <$60k. F-150 Lightning/R1T no tienen volumen. Nicho vacío.

6. CANIBALIZACIÓN PROPIA: Tesla Model S quedó en "Segmento residual". Si no matas tu producto viejo, el mercado lo hace.
"""

ax4.text(0.01, 0.95, recomendaciones, transform=ax4.transAxes, fontsize=12,
         va="top", ha="left", fontfamily="monospace",
         bbox=dict(boxstyle="round", facecolor="#1a1a1a", edgecolor="#00A699", linewidth=2, alpha=0.9))

# Pie de página
fig.text(0.5, 0.01, "Fuente: WA DOL Vehicle Registration Database 2012-2023 | Modelos: K-Means K=3 Silhouette 0.454 + Random Forest AUC 0.993",
         ha="center", fontsize=9, color="gray", style="italic")

plt.tight_layout(rect=[0, 0.02, 1, 0.97])
plt.show()

**Conclusión final:**

El análisis descriptivo y el uso de algoritmos K-Means y Random Forest convergen en los siguientes puntos esenciales para entrar al mercado de vehículos eléctricos:
* Un cambio tecnológico que apunta directamente al uso de tecnologías de mecanismos autónomos eléctricos.
* Severas dificultades de otras marcas, que no sean Tesla, para consolidarse en el mercado.

**Punto de oportunidad:**
* Aprovechar tendencia del mercado actual y potenciar la venta y distribución de vehículos BEV con autonomía de ≥ 250 millas (modelos más recientes).

**Nota final:** un posterior análisis podría indagar sobre nichos específicos que las diferentes marcas han logrado consolidar para no salir definitivamente del mercado. Algún modelo en especial o tipo de vehículo que les permite conseguir ventas en un mercado tan controlado.